## Upload a Custom Random Image Dataset to S3

### Objective
Create a synthetic image dataset with 100 random images, upload every image to S3 through LAILA, and store the `{name: global_id}` mapping as a manifest entry named `my_dataset_manifest`.


In [1]:
%load_ext autoreload
%autoreload 2

### Objective
Import dependencies, configure paths, and define the constants used by the example.


In [2]:
import sys
from pathlib import Path

import numpy as np
from PIL import Image

PROJECT_ROOT = Path("/home/ubuntu/laila")
PYTHON_ROOT = PROJECT_ROOT.parent
if str(PYTHON_ROOT) not in sys.path:
    sys.path.append(str(PYTHON_ROOT))

import laila
laila.read_args(PROJECT_ROOT / "vault" / "dev_secrets.toml")
from laila.pool import S3Pool

POOL_NICKNAME = "custom_image_s3_pool"
MANIFEST_NICKNAME = "my_dataset_manifest"
IMAGE_COUNT = 100
IMAGE_SIZE = (64, 64)


/home/ubuntu/laila/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Objective
Define the helper for constructing the S3 pool used by this example.


In [3]:
def build_s3_pool() -> S3Pool:
    required = ["AWS_BUCKET_NAME", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION"]
    missing = [name for name in required if getattr(laila.args, name, None) is None]
    if missing:
        raise RuntimeError(f"Missing laila.args values: {', '.join(missing)}")

    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
    )

s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=POOL_NICKNAME)


### Objective
Register the S3 pool, generate 100 random RGB images with deterministic nicknames, and build the `{name: global_id}` manifest dictionary.


In [4]:
rng = np.random.default_rng(42)
image_entries = []
manifest_dict = {}

for j in range(IMAGE_COUNT):
    image_name = f"image_{j}"
    image_array = rng.integers(0, 256, size=(*IMAGE_SIZE, 3), dtype=np.uint8)

    # Validate the array as an actual image before wrapping it with LAILA.
    new_image = Image.fromarray(image_array, mode="RGB")

    image_entry = laila.constant(data=new_image, nickname=image_name)
    image_entries.append(image_entry)
    manifest_dict[image_name] = image_entry.global_id

print("Generated image entries:", len(image_entries))
print("First item:", next(iter(manifest_dict.items())))

Generated image entries: 100
First item: ('image_0', 'LAILA:ENTRY:GLOBAL_ID:b0f18c7b-bb24-561d-a287-f1b20c35a2f5')


### Objective
Upload the full image dataset to S3 through one batched LAILA call.


In [5]:
upload_group_future = laila.memorize(image_entries, pool_nickname=POOL_NICKNAME)
print("Upload future type:", type(upload_group_future).__name__)
print("Status before wait:", upload_group_future.status)

upload_group_future.wait()
print("Status after wait:", upload_group_future.status)
print("Uploaded entries:", len(upload_group_future))

Upload future type: GroupFuture
Status before wait: {'total': 100.0, 'percentages': {'finished': 0.0, 'running': 0.0, 'not_started': 100.0, 'error': 0.0, 'cancelled': 0.0}}
Status after wait: {'total': 100.0, 'percentages': {'finished': 100.0, 'running': 0.0, 'not_started': 0.0, 'error': 0.0, 'cancelled': 0.0}}
Uploaded entries: 100


### Objective
Persist the `{name: global_id}` manifest itself as a deterministic LAILA entry named `my_dataset_manifest`.


In [6]:
manifest_entry = laila.constant(data=manifest_dict, nickname=MANIFEST_NICKNAME)
manifest_future = laila.memorize(manifest_entry, pool_nickname=POOL_NICKNAME)
manifest_future.wait()

print("Manifest nickname:", MANIFEST_NICKNAME)
print("Manifest global_id:", manifest_entry.global_id)
print("Manifest size:", len(manifest_dict))
print("Sample manifest rows:")
for index, item in enumerate(manifest_dict.items()):
    if index >= 5:
        break
    print(item)

Manifest nickname: my_dataset_manifest
Manifest global_id: LAILA:ENTRY:GLOBAL_ID:75a25c3d-bc58-533e-a068-4315c2d28c51
Manifest size: 100
Sample manifest rows:
('image_0', 'LAILA:ENTRY:GLOBAL_ID:b0f18c7b-bb24-561d-a287-f1b20c35a2f5')
('image_1', 'LAILA:ENTRY:GLOBAL_ID:5cbfe8fc-8194-57ca-99fc-799de7dadf84')
('image_2', 'LAILA:ENTRY:GLOBAL_ID:854d6d65-81ea-524e-86a3-d3a86d4d4330')
('image_3', 'LAILA:ENTRY:GLOBAL_ID:ce341496-45ca-500c-947c-771ead17e577')
('image_4', 'LAILA:ENTRY:GLOBAL_ID:76b55fbd-8399-5845-8400-7431258ffaf9')
